# Tutorial: Running the Optimized 4D Whole Cell Model on the Delta Gateway

**RESOURCES**

**Primary reference:** [Thornburg Z et. al, Bringing the genetically minimal cell to life on a computer in 4D, Cell 2026](https://www.cell.com/cell/fulltext/S0092-8674(26)00174-1)

**Code (optimized production fork):** [luthey-schulten-chemistry/Optimize_4DWCM_Minimal_Cell](https://github.com/luthey-schulten-chemistry/Optimize_4DWCM_Minimal_Cell)

**About:** This notebook launches the **optimized** 4DWCM (~3× faster than the paper baseline) inside the Gateway **4DCell Optimized** container.

**Website:** [4D Minimal Cell](https://minimalcell4d.web.illinois.edu/home/)

**Gateway session requirements:**
- **GPU Environment:** `4DCell Optimized`
- **GPUs:** 2 (RDME/CME on GPU 0, async DNA on GPU 1)
- **Jupyter kernel:** `LM 2.5 (Python 3.7)` — kernel name is legacy; the container ships LM 2.6 + btree_chromo 2.0

**Before starting:** clone `Optimize_4DWCM_Minimal_Cell` into your workspace (not the old `Minimal_Cell_4DWCM` repo).


---
## 1. Setting paths and log directory

**One-time setup** (Terminal in Gateway, if not already cloned):

```bash
cd /home/user/workspace
git clone https://github.com/luthey-schulten-chemistry/Optimize_4DWCM_Minimal_Cell.git
```

Gateway layout: repo at **`/home/user/workspace/Optimize_4DWCM_Minimal_Cell`**. The next cell creates **`logs/`** and sets environment variables for the optimized stack.

In [ ]:
import os
from pathlib import Path

HOME = Path("/home/user/workspace")
REPO = HOME / "Optimize_4DWCM_Minimal_Cell"
LOGDIR = REPO / "logs"
LOGDIR.mkdir(parents=True, exist_ok=True)

# Optimized stack paths (match 4DCell Optimized container)
DNA_SOFTWARE = "/Software/opt"
PYLM_PATH = "/Software/Lattice_Microbes_2.6/src/pylm"

os.environ["TMPDIR"] = "/tmp"
os.environ["HOME"] = str(HOME)
os.environ["PATH"] = "/opt/conda/envs/lm_2.5_dev/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = PYLM_PATH + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib64:/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["DNA_GPU_ID"] = "1"          # async DNA on second GPU
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Short test defaults (edit for a production run)
OUTPUT_DIR = "gateway_test"
SIM_TIME = 10          # biological seconds; use 7200 for full cell cycle
RNG_SEED = 13
RDME_GPU = 0

print("REPO:", REPO)
print("LOGDIR:", LOGDIR)
print("DNA_SOFTWARE (-dsd):", DNA_SOFTWARE)
print("PYTHONPATH:", os.environ["PYTHONPATH"])

---
### Environment check (optional)

Runs **after** section 1. Package imports use `/opt/conda/envs/lm_2.5_dev/bin/python` (same as the simulation), not the Jupyter kernel — missing `jLM`/`pyLM` in the kernel alone is normal.

If **Hook.py** is missing, clone the repo in a Gateway terminal first (section 1).


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

SIM_PYTHON = "/opt/conda/envs/lm_2.5_dev/bin/python"
check_env = os.environ.copy()
check_env["PYTHONPATH"] = PYLM_PATH

print("=" * 60)
print("Environment check (4DCell Optimized)")
print("=" * 60)
print(f"\nJupyter kernel: {sys.executable} ({sys.version.split()[0]})")
print("  (packages below are checked with the simulation Python, not this kernel)")
print(f"Simulation Python: {SIM_PYTHON}")

print("\n--- Python packages (simulation env) ---")
if not os.path.isfile(SIM_PYTHON):
    print(f"  WARN: {SIM_PYTHON} not found — are you in 4DCell Optimized?")
else:
    for mod in ("jLM", "pyLM", "lm", "odecell"):
        cmd = [
            SIM_PYTHON, "-c",
            f"import {mod}; print(getattr({mod}, '__file__', 'ok'))",
        ]
        r = subprocess.run(cmd, env=check_env, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  {mod:8s} OK  ({r.stdout.strip()})")
        else:
            err = (r.stderr or r.stdout or "import failed").strip().splitlines()[-1]
            print(f"  {mod:8s} MISSING — {err}")

print("\n--- Optimized fork markers ---")
hook = REPO / "Hook.py"
if hook.is_file():
    txt = hook.read_text()
    print("  setSolverCached:", "setSolverCached" in txt)
    print("  Cython ODE on:  ", "_ode_use_cython = True" in txt)
else:
    print(f"  WARN: {hook} not found")
    print("  Clone first: git clone https://github.com/luthey-schulten-chemistry/Optimize_4DWCM_Minimal_Cell.git")

print("\n--- DNA / LAMMPS stack ---")
for label, paths in (
    ("btree_chromo", [f"{DNA_SOFTWARE}/btree_chromo/build/apps/btree_chromo"]),
    ("gen_sc_chain", [f"{DNA_SOFTWARE}/sc_chain_generation/fortran/gen_sc_chain",
                       "/Software/sc_chain_generation/fortran/gen_sc_chain"]),
    ("LAMMPS lib", ["/usr/local/lib/liblammps_delta_kokkos.so.0",
                     "/usr/local/lib64/liblammps_delta_kokkos.so.0"]),
):
    p = next((x for x in paths if os.path.isfile(x)), None)
    print(f"  {label:14s} {p or 'NOT FOUND'}")

print("\n--- GPUs ---")
nvsmi = shutil.which("nvidia-smi")
if nvsmi:
    subprocess.run([nvsmi, "-L"], check=False)
else:
    print("  nvidia-smi not on PATH (common in Jupyter); checking fallbacks")
    cvd = os.environ.get("CUDA_VISIBLE_DEVICES", "(unset)")
    print(f"  CUDA_VISIBLE_DEVICES: {cvd}")
    dev_nodes = sorted(Path("/dev").glob("nvidia*"))
    print(f"  /dev/nvidia* nodes: {', '.join(p.name for p in dev_nodes) or '(none)'}")
    drv = Path("/proc/driver/nvidia/version")
    if drv.is_file():
        print(f"  driver: {drv.read_text().splitlines()[0]}")
print("\n" + "=" * 60)

---
## 2. Create a log file

Rename the log file in the cell to store the output of the simulation we are going to run in the next cell. For instance **`logs/run.log`**.

**Note:** Make sure you don't overwrite the log file from prevoius run. The filenames should be different for each run.

In [ ]:
import subprocess

RUN_LOG = LOGDIR / "run_gateway_test.log"


---
## 3. Start the simulation

---
### Command-line arguments (`Whole_Cell_Minimal_Cell.py`)

| Variable | Shorthand | Description |
|----------|-----------|-------------|
| `--outputDir` | `-od` | Name of directory (under `Data/`) to store trajectories. |
| `--simTime` | `-t` | Biological time to simulate, in **seconds** (7200 = full cell cycle) |
| `--cudaDevices` | `-cd` | GPU index for RDME + CME (use **0**) |
| `--dnaSoftwareDirectory` | `-dsd` | DNA software root — use **`/Software/opt/`** in 4DCell Optimized |
| `--dnaRngSeed` | `-drs` | Integer RNG seed for the chromosome / DNA pipeline |
| `--maximumHours` | `-mh` | Optional wall-clock limit (hours); checkpoints then stops cleanly |
| `--workingDirectory` | `-wd` | Base directory of the run (default: current working directory) |

**Environment (set in section 1):**
- `PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm`
- `DNA_GPU_ID=1` — btree_chromo runs on the **second** GPU (async overlap)
- `LD_LIBRARY_PATH` must include `/usr/local/lib` (Kokkos LAMMPS)

**Example (gateway, background + log):**

```bash
export PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm
export DNA_GPU_ID=1
export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib
nohup python -u Whole_Cell_Minimal_Cell.py \
  -od gateway_test -t 10 -cd 0 -drs 13 -dsd /Software/opt/ \
  > logs/run.log 2>&1 &
```

Edit **`OUTPUT_DIR`**, **`SIM_TIME`**, and **`RUN_LOG`** in the cells above/below. This notebook runs a **short test** by default (`SIM_TIME=10`). A full 7200 s cell cycle takes **~25 h wall** on 2× A100 with the optimized stack (not 3–4 days).

> **Gateway session limit:** interactive Jupyter sessions are short. Use a **batch job** (Slurm) for production 7200 s runs; pass `-mh` slightly below your wall limit so `restart_files/` are written before the job ends.

In [ ]:
%%bash
set -e
REPO=/home/user/workspace/Optimize_4DWCM_Minimal_Cell
ENTRY="$REPO/Whole_Cell_Minimal_Cell.py"

if [ ! -f "$ENTRY" ]; then
  echo "ERROR: $ENTRY not found."
  echo "Clone first:"
  echo "  cd /home/user/workspace"
  echo "  git clone https://github.com/luthey-schulten-chemistry/Optimize_4DWCM_Minimal_Cell.git"
  exit 1
fi

mkdir -p "$REPO/logs"
cd "$REPO"

export PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm
export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib:${LD_LIBRARY_PATH:-}
export DNA_GPU_ID=1
export TMPDIR=/tmp
export HOME=/home/user/workspace
export HDF5_USE_FILE_LOCKING=FALSE
export XDG_CACHE_HOME=/tmp/.cache
export PYTHONPYCACHEPREFIX=/tmp/.pycache

nohup /opt/conda/envs/lm_2.5_dev/bin/python -u "$ENTRY" \
  -od gateway_test -t 10 -cd 0 -drs 13 -dsd /Software/opt/ \
  > logs/run_gateway_test.log 2>&1 &
echo "Simulation started in background, PID: $!"
echo "Log: logs/run_gateway_test.log"

---
## 4. Track the progress

A **7200 s** optimized run takes **~25 h wall** on 2× A100 (vs. ~78 h for the unoptimized baseline). The first metabolism hook compiles the Cython ODE once (~60–90 s); later hooks are ~1 s each.

**Healthy log markers:**
- `CME_WORKER: persistent worker is ready`
- `Compiling cythonCompiledFunctions.pyx` (once)
- `ODE time: ~1` (after the first hook)
- `DNA time: ~0.5` s, no repeated `RESCUE`

The cell below prints the last 80 lines of the log file.


In [ ]:
if RUN_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(RUN_LOG)], check=False)
else:
    print(f"No log yet: {RUN_LOG}\n")

---
## 5. Restart the simulation (optional)

Run **only if the previous simulation failed** or you want to continue toward 7200 s after a checkpointed stop (`-mh`). Resumes from `Data/{output_dir}/` using **`Restart_Whole_Cell_Minimal_Cell.py`**.

Use the **same** `-od`, `-dsd /Software/opt/`, `PYTHONPATH`, and `DNA_GPU_ID=1` as the original run. `-t` is **additional** biological seconds (e.g. if you reached 3600 s and want 7200 s total, pass `-t 3600`).

**Quick check it actually failed:** run the tail cell above; if the last line shows a traceback and `pgrep -f Whole_Cell_Minimal_Cell.py` returns nothing, the run is dead.

In [ ]:
import subprocess

RESTART_LOG = LOGDIR / "restart_gateway_test.log"


In [ ]:
%%bash
set -e
cd /home/user/workspace/Optimize_4DWCM_Minimal_Cell

if pgrep -f "Whole_Cell_Minimal_Cell.py" > /dev/null; then
  echo "ERROR: a Whole_Cell_Minimal_Cell.py process is still running."
  pgrep -af "Whole_Cell_Minimal_Cell.py" || true
  exit 1
fi

export PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm
export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib:${LD_LIBRARY_PATH:-}
export DNA_GPU_ID=1
export TMPDIR=/tmp
export HOME=/home/user/workspace
export HDF5_USE_FILE_LOCKING=FALSE

nohup /opt/conda/envs/lm_2.5_dev/bin/python -u Restart_Whole_Cell_Minimal_Cell.py \
  -od gateway_test -t 10 -cd 0 -drs 13 -dsd /Software/opt/ \
  > logs/restart_gateway_test.log 2>&1 &
echo "Restart started in background, PID: $!"

In [ ]:

if RESTART_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(RESTART_LOG)], check=False)
else:
    print(f"No log yet: {RESTART_LOG}\n")

---

## 6. Cancel the simulation jobs

Run the next cell to list running simulation PIDs, then set **`JOB_PID`** in the following cell to send SIGTERM.

In [ ]:
%%bash
echo "Running jobs"
pgrep -af "Whole_Cell_Minimal_Cell.py" || echo "(none)"


In [ ]:
import subprocess

JOB_PID = 0  # set to integer PID from pgrep cell above, then re-run

if not JOB_PID:
    print("Set JOB_PID to an integer (the process id), then re-run this cell.")
else:
    r = subprocess.run(["kill", str(JOB_PID)], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"Sent SIGTERM to PID {JOB_PID}.")
    else:
        print(f"kill failed (code {r.returncode}): {r.stderr or r.stdout or 'no such process or permission denied'}")